# Cat Health Agentic RAG with LangChain and LangGraph

In Session 1, retrieval happened before every answer:

```text
question -> retrieve -> generate
```

In this notebook, retrieval becomes a **tool**. The agent can call that tool when it decides the user's question needs cat health guideline context.

That is the core idea of agentic RAG for this session:

```text
question -> agent decides whether to retrieve -> optional retrieval tool call -> answer
```

We will show that loop two ways:

1. **High-level LangChain path**: use `create_agent` with middleware.
2. **Explicit LangGraph path**: build the same loop with `StateGraph`, `ToolNode`, and `tools_condition`.

Both versions use the same retriever tool. The point is to see that agentic RAG is about giving the agent retrieval as an action, not forcing retrieval as a pre-step.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how agentic RAG differs from a fixed two-step RAG pipeline.
- Build and inspect a retrieval tool over a Qdrant vector store.
- Use LangChain middleware to observe and constrain an agent loop.
- Compare the convenience of `create_agent` with the control of an explicit LangGraph graph.
- Design focused routing and middleware experiments for an agentic RAG system.

## Table of Contents

- **Breakout Room #1: High-Level Agentic RAG with LangChain**
  - Task 1: Environment Setup
  - Task 2: Load and Index the Cat Health Corpus
  - Task 3: Create a Retriever Tool
  - Task 4: Build an Agent with `create_agent` and Middleware
  - Task 5: Visualize and Stream the `create_agent` Agent
  - 🏗️ Activity #1: Add a Retriever Tool-Call Budget
- **Breakout Room #2: Explicit Agent Loop with LangGraph**
  - Task 6: Build the Same Agent Loop with LangGraph
  - 🏗️ Activity #2: Add Deterministic Scope Routing
  - 🚧 Advanced Build: Add Explicit Retrieval Quality Control

---
# Breakout Room #1
## High-Level Agentic RAG with LangChain

In this breakout room, you will build the shared retrieval tool, give it to `create_agent`, and use middleware and streaming to inspect and constrain the agent loop.

## Task 1: Environment Setup

From the `02_Agentic_RAG_LangGraph_LangChain` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain gives us document loading, splitting, embeddings, Qdrant integration, tools, models, and the high-level agent loop.

In [ ]:
from pathlib import Path
from getpass import getpass
import os

from IPython.display import Image, display

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware, before_model
from langchain.tools import tool
from langchain_community.document_loaders import TextLoader
from langchain_core.messages import SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

### API Keys and Models

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

LangSmith tracing is optional. If you set `LANGSMITH_TRACING=true` and `LANGSMITH_API_KEY`, LangChain/LangGraph calls will be traced automatically.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

os.environ.setdefault("LANGSMITH_PROJECT", "aim-session-3-agentic-rag")

chat_model_name = os.environ.get("AIM_CHAT_MODEL", "gpt-5.4-mini")
embedding_model_name = os.environ.get("AIM_EMBEDDING_MODEL", "text-embedding-3-small")

llm = ChatOpenAI(model=chat_model_name)
embeddings = OpenAIEmbeddings(model=embedding_model_name)

print(f"Chat model: {chat_model_name}")
print(f"Embedding model: {embedding_model_name}")
print(f"LangSmith tracing: {os.environ.get('LANGSMITH_TRACING', 'false')}")

## Task 2: Load and Index the Cat Health Corpus

We will use a small course-owned Markdown corpus instead of a PDF. This keeps the session focused on the agentic RAG pattern instead of PDF parsing.

**Further Reading:**
- [LangChain Retrieval](https://docs.langchain.com/oss/python/langchain/retrieval)
- [Qdrant LangChain Integration](https://qdrant.tech/documentation/frameworks/langchain/)

In [ ]:
corpus_path = Path("data/cat_health_guidelines.md")

if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health corpus at: {corpus_path.resolve()}\n"
        "Run this notebook from the 02_Agentic_RAG_LangGraph_LangChain folder."
    )

loader = TextLoader(str(corpus_path), encoding="utf-8")
documents = loader.load()

for document in documents:
    document.metadata["source"] = corpus_path.name
    document.metadata["document_type"] = "cat_health_guidelines"

print(f"Loaded {len(documents)} document(s).")
print(documents[0].page_content[:800])
print("\nMetadata:", documents[0].metadata)

### Split the Corpus

Chunks should be large enough to keep a useful idea together, but small enough that retrieval returns focused context.

In [ ]:
chunk_size = 900
chunk_overlap = 120

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " "],
)

splits = text_splitter.split_documents(documents)

for index, split in enumerate(splits):
    split.metadata["chunk_id"] = index

print(f"Created {len(splits)} chunks.")
print(splits[0].page_content[:800])
print("\nMetadata:", splits[0].metadata)

### Build the Qdrant Vector Store

For the course notebook, Qdrant runs in memory. There is no Docker service or cloud account required, and the collection disappears when the notebook kernel stops.

In [ ]:
collection_name = "cat_health_agentic_rag"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
)

retrieval_k = 4
print(f"Built in-memory Qdrant collection: {collection_name}")

## Task 3: Create a Retriever Tool

This is the important shift from Session 1.

The retriever is no longer a required pre-step. It is now a tool the agent can call when it wants context from the cat health guideline corpus.

The tool name, docstring, inputs, and output format form a contract with the model. Clear contracts make good tool-selection and grounded-answer behavior more likely.

**Further Reading:**
- [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)

In [ ]:
def _format_retrieved_docs(scored_docs: list[tuple]) -> str:
    formatted_chunks = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        source = doc.metadata.get("source", "unknown")
        chunk_id = doc.metadata.get("chunk_id", "unknown")
        start_index = doc.metadata.get("start_index", "unknown")
        score_text = f"{score:.3f}" if isinstance(score, (float, int)) else str(score)
        formatted_chunks.append(
            f"[Source {index}: {source}, chunk_id={chunk_id}, start_index={start_index}, score={score_text}]\n"
            f"{doc.page_content.strip()}"
        )
    return "\n\n".join(formatted_chunks)


@tool
def retrieve_cat_health_guidelines(query: str) -> str:
    """Search the cat health guideline corpus for relevant context about cat preventive care, nutrition, hydration, vaccines, parasites, dental health, urinary warning signs, emergencies, senior cats, stress, behavior, and safe home monitoring."""
    results = vector_store.similarity_search_with_score(query, k=retrieval_k)
    if not results:
        return "No relevant cat health guideline context found."
    return _format_retrieved_docs(results)


retriever_tool = retrieve_cat_health_guidelines

Try the tool directly once. This is just to understand what the agent will see when it calls the tool.

In [ ]:
print(
    retriever_tool.invoke(
        {"query": "What urinary signs suggest a cat needs urgent veterinary care?"}
    )[:2500]
)

#### ❓ Question #1

What changes when retrieval becomes a tool instead of a mandatory first step?

##### Answer:

The shift is from a fixed two-step pipeline ("retrieve, then generate") to an agent loop where retrieval is one *option* the model can take per turn. The model decides per question whether the corpus is needed at all, and can issue multiple differently-worded retrieval calls in a single run.

**Benefits:**
- Cheaper and faster for out-of-scope questions — no embedding or vector search happens at all when the model decides the corpus isn't relevant (in my run, the FIFA question never triggered a tool call).
- More flexible for compound questions — the agent can search for sub-topics separately rather than cramming everything into one query.
- Retrieval decisions become visible and auditable — streaming the graph shows exactly when and why a tool call was made.

**Costs:**
- Non-determinism: the model might skip retrieval when it should retrieve, or retrieve on a vague match. The tool name, docstring, and system prompt now do a lot of work to steer the choice.
- More moving parts to debug — the agent can loop, make multiple calls, or stop early, so you have to inspect intermediate events instead of judging only the final answer.


## Task 4: Way 1 - Build an Agent with `create_agent` and Middleware

`create_agent` builds the agent loop for us:

1. The model reads the user question and available tools.
2. The model either answers directly or asks to call a tool.
3. If it asks for a tool, LangChain executes the tool.
4. The tool result is added back to the message history.
5. The model continues until it produces a final answer.

This is the fastest way to build agentic RAG: give the agent a retriever tool and let the agent decide when to call it.

Middleware hooks into the loop without requiring us to rebuild the graph. Below, custom `before_model` middleware logs each model step, while built-in middleware limits the number of model calls in one run.

**Further Reading:**
- [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain Middleware](https://docs.langchain.com/oss/python/langchain/middleware)

In [ ]:
AGENT_SYSTEM_PROMPT = """You are a cat health guideline assistant in an agentic RAG lesson.

You have one retrieval tool: retrieve_cat_health_guidelines.

Use the retrieval tool when the user asks about cat health, cat symptoms, preventive care, nutrition, vaccines, parasites, dental health, urinary signs, senior cats, stress, behavior, or home monitoring.

When you use retrieved context:
- Answer only from that retrieved context.
- Include a short Sources line using the source labels returned by the tool.
- Remind the user to contact a veterinarian for medical decisions, urgent symptoms, or worsening symptoms.

If the user asks something unrelated to cat health, do not call the tool. Briefly say this notebook is scoped to the cat health guideline corpus.
If the retrieved context does not contain enough information, say you do not have enough information in the cat health guidelines to answer.
"""


@before_model
def log_before_model(state, runtime):
    """Log a compact view of each model step in the agent loop."""
    print(f"[middleware] Calling the model with {len(state['messages'])} message(s).")


agent_middleware = [
    log_before_model,
    ModelCallLimitMiddleware(run_limit=4, exit_behavior="end"),
]


agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=AGENT_SYSTEM_PROMPT,
    middleware=agent_middleware,
)

print(type(agent))

#### ❓ Question #2

What does middleware let us change or observe without rebuilding the agent loop? Why is a model-call limit useful?

##### Answer:

Middleware hooks into the agent loop at well-defined points (`before_model`, `after_model`, tool-call wrappers) without forcing us to rebuild the underlying graph. In this notebook I used it to:

- **Observe** what the loop is doing — my `log_before_model` printed "Calling the model with N message(s)" each step, so I could see the message-history growing as the agent made tool calls.
- **Constrain** the loop — `ModelCallLimitMiddleware(run_limit=4)` caps the number of model invocations per run.
- **Limit specific tools** — `ToolCallLimitMiddleware` (used in Activity #1) caps how many times a single tool can be called.

A model-call limit is essentially a safety belt. Tool-using agents can loop indefinitely if the model keeps issuing new tool calls — a confused model, a malformed tool result, or an adversarial input can lead to runaway costs and latency. A hard cap on model invocations means even a worst-case run terminates and the bill stays bounded.


## Task 5: Visualize and Stream the `create_agent` Agent

`create_agent` returns a compiled LangGraph graph. We can visualize that graph and stream updates to inspect when the retriever tool was called.

The exact generated graph includes middleware nodes, but its core loop is:

```text
START -> before-model middleware -> model -> after-model middleware -> END
                                                |
                                                | tool call
                                                v
                                              tools
                                                |
                                                +----> loop back to before-model middleware
```

**Further Reading:**
- [LangGraph Streaming](https://docs.langchain.com/oss/python/langgraph/streaming)
- [LangSmith Observability](https://docs.langchain.com/langsmith/observability)

### Visualize the `create_agent` Graph

Run the next cell to render the exact compiled graph, including middleware nodes. If Mermaid PNG rendering is unavailable in your environment, the fallback prints the Mermaid source.

In [ ]:
try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Could not render Mermaid PNG. Mermaid source follows:\n")
    print(agent.get_graph().draw_mermaid())
    print(f"\nRender error: {exc}")

### Stream Agent Runs

Streaming updates lets us inspect the path the agent took. Look for tool messages to see when retrieval happened.

In [ ]:
def print_agent_stream(question: str):
    """Run the agent and print each graph update."""
    inputs = {"messages": [{"role": "user", "content": question}]}

    for chunk in agent.stream(inputs, stream_mode="updates"):
        for node_name, update in chunk.items():
            print(f"\n--- Update from node: {node_name} ---")
            if update is None:
                print("No state update returned.")
                continue

            messages = update.get("messages", [])
            if not messages:
                print(update)
                continue

            latest_message = messages[-1]
            if isinstance(latest_message, ToolMessage):
                print(f"Tool result preview:\n{latest_message.content[:1200]}")
            elif hasattr(latest_message, "pretty_print"):
                latest_message.pretty_print()
            else:
                print(latest_message)

### Example 1: Cat Health Question

This should call the retrieval tool before answering.

In [ ]:
print_agent_stream("What urinary signs suggest my cat needs urgent veterinary care?")

### Example 2: Another Cat Health Question

This should also retrieve, but with a different search query.

In [ ]:
print_agent_stream("What preventive care should an adult cat get each year?")

### Example 3: Unrelated Question

This should not call the retrieval tool.

In [ ]:
print_agent_stream("Who won the 2022 FIFA World Cup?")

#### ❓ Question #3

For each example, did the agent call the retrieval tool? Why or why not?

##### Answer:

- **Example 1 — "What urinary signs suggest my cat needs urgent veterinary care?"** — Yes. The streamed updates showed a `tools` node update with a call to `retrieve_cat_health_guidelines`, and the model used the returned chunks (urinary warning signs + emergency symptoms) to answer. The question is exactly the kind of in-scope topic listed in both the tool docstring and the system prompt.
- **Example 2 — "What preventive care should an adult cat get each year?"** — Yes. Same pattern: the agent issued a retrieval call with a preventive-care-flavored query, got the relevant chunks, and grounded the answer in them. Again, "preventive care" is in the explicit scope list.
- **Example 3 — "Who won the 2022 FIFA World Cup?"** — No. The model went straight to a final answer ("This notebook is scoped to the cat health guideline corpus, so I can't answer that question here.") without ever hitting the `tools` node. The system prompt explicitly tells the agent not to call the tool for unrelated questions, and the model correctly classified this one as off-topic.

The tool-call decision is driven by three things: the tool name/docstring (which advertises what the tool is good for), the system prompt's in-scope list, and the model's own classification of the user's question against both.


## 🏗️ Activity #1: Add a Retriever Tool-Call Budget

Middleware can enforce an operational rule without changing the retriever tool or rebuilding the agent graph. Create a second agent that allows at most **one** call to `retrieve_cat_health_guidelines` per run.

### Requirements

1. Create a `ToolCallLimitMiddleware` instance scoped to `retriever_tool.name` with `run_limit=1` and `exit_behavior="continue"`.
2. Create `budgeted_agent` with the same model, tool, prompt, and existing middleware plus the new retrieval budget.
3. Ask the agent to use separate searches for urinary emergency signs and annual preventive care before summarizing both.
4. Inspect the stream and explain what the middleware allowed or blocked.

**Further Reading:**
- [Built-in Middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in)

In [ ]:
# Activity #1: Add a Retriever Tool-Call Budget
# Build a second agent that allows at most ONE call to the retriever per run.
# With exit_behavior="continue", the agent keeps running after hitting the limit
# but the retriever can no longer be called — it has to summarize from what it
# already has.

retrieval_budget = ToolCallLimitMiddleware(
    tool_name=retriever_tool.name,
    run_limit=1,
    exit_behavior="continue",
)

budgeted_agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=AGENT_SYSTEM_PROMPT,
    middleware=agent_middleware + [retrieval_budget],
)


def print_budgeted_stream(question: str):
    """Run the budgeted agent and print each graph update."""
    inputs = {"messages": [{"role": "user", "content": question}]}
    for chunk in budgeted_agent.stream(inputs, stream_mode="updates"):
        for node_name, update in chunk.items():
            print(f"\n--- Update from node: {node_name} ---")
            if update is None:
                print("No state update returned.")
                continue
            messages = update.get("messages", [])
            if not messages:
                print(update)
                continue
            latest_message = messages[-1]
            if isinstance(latest_message, ToolMessage):
                print(f"Tool result preview:\n{latest_message.content[:1200]}")
            elif hasattr(latest_message, "pretty_print"):
                latest_message.pretty_print()
            else:
                print(latest_message)


# Compound question that ideally wants two separate retrieval calls:
# one for urinary emergencies and one for annual preventive care.
# With run_limit=1, the agent will only get to do one of them.
print_budgeted_stream(
    "Please use separate searches: first look up urinary emergency warning signs, "
    "then look up annual preventive care for an adult cat. "
    "Summarize both topics with sources."
)


### 📝 Activity #1 Notes

##### Answer:

- **Which retrieval calls did the agent attempt?** The model first issued a retrieval call for urinary emergency warning signs. After it got those chunks back, it tried a second retrieval call for annual preventive care.
- **Which call did the middleware allow or block?** The first urinary-emergency call went through normally. The second preventive-care call was intercepted by `ToolCallLimitMiddleware(run_limit=1)` — instead of being executed, the middleware returned a budget-exhausted message to the model. With `exit_behavior="continue"`, the agent kept running but had to summarize using only the urinary context it already had (and whatever general knowledge it brings without a corpus).
- **What quality or safety trade-off does this budget introduce?** A tool-call budget is a hard cost/latency guardrail — you can't accidentally rack up many embedding/search calls in one run, and an adversarial prompt that tries to flood the agent with tool calls is contained. The trade-off is answer quality: a genuinely compound question that needs multiple targeted searches will be answered from incomplete context. In my run the urinary section was strong and well-sourced, but the preventive-care section became thinner because that retrieval never happened. The right limit depends on the worst-case run you want to bound: stricter limits = safer cost ceiling but more answers that are partially un-grounded.


## Breakout Room #1 Summary

In BOR1, you:

- Turned retrieval into a source-labeled tool the model can choose to call.
- Built a high-level agent loop with `create_agent`.
- Used middleware to observe the loop and constrain model or tool calls.
- Used streaming to inspect retrieval decisions instead of judging only the final answer.

---
# Breakout Room #2
## Explicit Agent Loop with LangGraph

In this breakout room, you will rebuild the same model-tools loop explicitly, then add routing behavior that would require graph-level control.

## Task 6: Way 2 - Build the Same Agent Loop with LangGraph

Now we will build the minimal agent loop ourselves.

This is the same idea as `create_agent`, but expressed directly as a graph:

```text
START -> agent model -------------------------------> END
              |
              | tool call
              v
            tools
              |
              +---------------------> agent model
```

There is still no mandatory pre-retrieval step. Retrieval only happens if the model emits a tool call. Unlike middleware, graph nodes and conditional edges let us change the control flow itself.

**Further Reading:**
- [LangGraph Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api)
- [LangGraph Workflows and Agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)

In [ ]:
llm_with_tools = llm.bind_tools([retriever_tool])


def call_model(state: MessagesState):
    """Call the model with tools bound so it can choose whether to retrieve."""
    response = llm_with_tools.invoke(
        [SystemMessage(content=AGENT_SYSTEM_PROMPT)] + state["messages"]
    )
    return {"messages": [response]}


langgraph_builder = StateGraph(MessagesState)
langgraph_builder.add_node("agent", call_model)
langgraph_builder.add_node("tools", ToolNode([retriever_tool], handle_tool_errors=True))

langgraph_builder.add_edge(START, "agent")
langgraph_builder.add_conditional_edges("agent", tools_condition)
langgraph_builder.add_edge("tools", "agent")

langgraph_agent = langgraph_builder.compile()
print("Compiled the explicit LangGraph agent loop.")

### Visualize the Explicit LangGraph Agent

This graph should look like the core agent loop: model node, tools node, and a conditional route between them.

In [ ]:
try:
    display(Image(langgraph_agent.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Could not render Mermaid PNG. Mermaid source follows:\n")
    print(langgraph_agent.get_graph().draw_mermaid())
    print(f"\nRender error: {exc}")

### Stream the Explicit LangGraph Agent

This helper is intentionally similar to `print_agent_stream`. The difference is that we are streaming from the graph we built ourselves.

In [ ]:
def print_langgraph_stream(question: str):
    """Run the explicit LangGraph agent and print each graph update."""
    inputs = {"messages": [{"role": "user", "content": question}]}

    for chunk in langgraph_agent.stream(inputs, stream_mode="updates"):
        for node_name, update in chunk.items():
            print(f"\n--- Update from node: {node_name} ---")
            messages = update.get("messages", [])
            if not messages:
                print(update)
                continue

            latest_message = messages[-1]
            if isinstance(latest_message, ToolMessage):
                print(f"Tool result preview:\n{latest_message.content[:1200]}")
            elif hasattr(latest_message, "pretty_print"):
                latest_message.pretty_print()
            else:
                print(latest_message)

### Compare the Same Questions

Run the same style of questions through the explicit LangGraph agent. The exact wording may differ, but the retrieval decision should follow the same pattern.

In [ ]:
print_langgraph_stream("What urinary signs suggest my cat needs urgent veterinary care?")

In [ ]:
print_langgraph_stream("Who won the 2022 FIFA World Cup?")

#### ❓ Question #4

What parts did `create_agent` hide that the explicit LangGraph version made visible? When would you choose middleware, and when would you change the graph itself?

##### Answer:

`create_agent` hides:
- The actual graph construction — nodes for `model` and `tools`, the entry edge from `START`, and the conditional routing back to the model.
- The decision logic for "should we go to tools or end?" — it embeds `tools_condition` internally.
- Where the middleware nodes get spliced in (before / after model).

The explicit LangGraph version made all of those visible because I had to write them:
- I defined `call_model` myself, so I controlled exactly what messages get prepended (the system prompt) and how the tool-bound LLM was invoked.
- I wired the edges explicitly: `START → agent`, conditional `agent → tools | END`, and `tools → agent`.
- The compiled graph diagram showed a clean two-node loop with no middleware wrappers around it.

**When to choose which:**
- **Middleware** is right for cross-cutting concerns that don't change control flow — logging, rate limits, tool budgets, retries, guardrails. Quick to add to a pre-built loop. Use it when the *shape* of the agent loop is fine but you want to observe or constrain it.
- **Change the graph itself** when you need to *restructure* the flow — add a deterministic pre-router (Activity #2), insert a relevance grader after retrieval, branch to a query-rewrite node when scores are low, or add a totally separate path for one class of inputs. These can't be cleanly added as middleware because they introduce new nodes and new edges.

Rule of thumb that worked for me: middleware = behavior, graph edits = topology.


## 🏗️ Activity #2: Add Deterministic Scope Routing

The base LangGraph sends every question to the model and relies on the model to reject unrelated requests. Add a small deterministic route before the agent so clearly unrelated questions can bypass the model-tools loop.

### Requirements

1. Add an `out_of_scope` node that returns a brief scope message.
2. Add a routing function that sends likely cat-health questions to `agent` and clearly unrelated questions to `out_of_scope`.
3. Build and compile a new graph with the route immediately after `START`.
4. Test at least one cat-health question, one unrelated question, and one ambiguous question.
5. Explain where the deterministic route helps and where it is brittle.

**Further Reading:**
- [LangGraph Conditional Edges](https://docs.langchain.com/oss/python/langgraph/graph-api#conditional-edges)

In [ ]:
# Activity #2: Add Deterministic Scope Routing
# Add a keyword-based pre-router. Clearly off-topic questions skip the
# model/tools loop entirely and get a short scope message instead.

from langchain_core.messages import AIMessage, HumanMessage

# Keywords that strongly signal a cat-health question.
CAT_HEALTH_KEYWORDS = {
    "cat", "cats", "kitten", "kittens", "feline",
    "vet", "veterinarian", "veterinary",
    "litter", "urinary", "urine", "urinate",
    "vaccine", "vaccines", "vaccination",
    "parasite", "parasites", "flea", "fleas", "tick",
    "dental", "tooth", "teeth", "gums",
    "grooming", "vomit", "vomiting", "diarrhea",
    "appetite", "hydration", "nutrition",
    "senior cat", "behavior", "stress",
}


def scope_route(state: MessagesState) -> str:
    """Send likely cat-health questions to the agent; send clearly unrelated questions to out_of_scope."""
    last_user_text = ""
    for message in reversed(state["messages"]):
        if isinstance(message, HumanMessage):
            last_user_text = message.content.lower()
            break

    if any(keyword in last_user_text for keyword in CAT_HEALTH_KEYWORDS):
        return "agent"
    return "out_of_scope"


def out_of_scope_node(state: MessagesState):
    """Return a short scope message without ever calling the model."""
    return {
        "messages": [
            AIMessage(
                content=(
                    "This notebook is scoped to the cat health guideline corpus. "
                    "I can only answer questions about cat preventive care, symptoms, "
                    "nutrition, vaccines, parasites, dental health, urinary warning signs, "
                    "and related topics."
                )
            )
        ]
    }


routed_builder = StateGraph(MessagesState)
routed_builder.add_node("agent", call_model)
routed_builder.add_node("tools", ToolNode([retriever_tool], handle_tool_errors=True))
routed_builder.add_node("out_of_scope", out_of_scope_node)

routed_builder.add_conditional_edges(
    START,
    scope_route,
    {"agent": "agent", "out_of_scope": "out_of_scope"},
)
routed_builder.add_conditional_edges("agent", tools_condition)
routed_builder.add_edge("tools", "agent")
routed_builder.add_edge("out_of_scope", END)

routed_agent = routed_builder.compile()


# Visualize the routed graph
try:
    display(Image(routed_agent.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Could not render Mermaid PNG. Mermaid source follows:\n")
    print(routed_agent.get_graph().draw_mermaid())
    print(f"\nRender error: {exc}")


def print_routed_stream(question: str):
    """Run the routed LangGraph agent and print each graph update."""
    print(f"\n>>> Question: {question}")
    inputs = {"messages": [{"role": "user", "content": question}]}
    for chunk in routed_agent.stream(inputs, stream_mode="updates"):
        for node_name, update in chunk.items():
            print(f"\n--- Update from node: {node_name} ---")
            messages = update.get("messages", [])
            if not messages:
                print(update)
                continue
            latest_message = messages[-1]
            if isinstance(latest_message, ToolMessage):
                print(f"Tool result preview:\n{latest_message.content[:1200]}")
            elif hasattr(latest_message, "pretty_print"):
                latest_message.pretty_print()
            else:
                print(latest_message)


# 1. On-topic cat health question → should go to agent → retrieve → answer
print_routed_stream("What urinary signs in a cat suggest urgent veterinary care?")

# 2. Clearly unrelated question → should bypass agent → out_of_scope
print_routed_stream("Who won the 2022 FIFA World Cup?")

# 3. Ambiguous: no explicit cat keyword but topically adjacent
print_routed_stream("How often should I take my pet for a checkup?")


### 📝 Activity #2 Notes

##### Answer:

- **Which questions bypassed the model-tools loop?** The clearly off-topic FIFA question was routed straight to `out_of_scope` without ever invoking the LLM. The ambiguous "How often should I take my pet for a checkup?" *also* bypassed — it doesn't contain any of the cat-health keywords, so the deterministic router treated it as off-topic even though a human would recognize it as adjacent. The on-topic urinary question matched the "cat" and "urinary" keywords and was sent through the normal agent → tools loop.
- **What happened with the ambiguous question?** It received the generic scope message even though the corpus actually has relevant content (preventive care frequency). This is the false-negative failure mode of pure keyword routing — synonyms ("kitty", "pet") and translations slip through.
- **What are the cost, latency, and quality trade-offs of this route?**
  - **Cost.** Clearly unrelated questions skip both the LLM call and any embedding/retrieval — that path is essentially free.
  - **Latency.** Same — no LLM round-trip, response is instant for the off-topic case.
  - **Quality.** The router is brittle. It mis-routes ambiguous inputs ("pet") to out-of-scope, and on the other side it can over-route to `agent` on inputs that mention "cat" in passing but aren't health-related ("Why does my cat hate Christmas trees?"). A production version would combine a deterministic pre-filter for *obvious* off-topic queries with the model-level system-prompt safety net (which we already saw correctly reject the FIFA question in Task 5) as the fallback for borderline cases.


## 🚧 Advanced Build (Optional): Add Explicit Retrieval Quality Control

The base assignment shows the minimal agentic RAG loop two ways:

1. `create_agent`
2. explicit LangGraph

If you want more control, extend the explicit LangGraph version with retrieval quality control. Good advanced additions include:

- Add a document relevance grader after retrieval.
- Add a query rewrite node when retrieval is weak.
- Add a loop limit so the agent cannot keep retrying forever.
- Add a deterministic guardrail before answering.

Those are useful production patterns, but they are not required for the core idea of agentic RAG.

---
## Summary

In this session, you:

1. Built a retrieval tool over a Qdrant vector store.
2. Used `create_agent` and middleware for a high-level agentic RAG loop.
3. Streamed agent runs to inspect when retrieval happened.
4. Rebuilt the loop explicitly with LangGraph nodes and conditional edges.
5. Practiced choosing between middleware-level constraints and graph-level routing.

### Key Takeaways

- Agentic RAG makes retrieval an available action instead of a mandatory pre-step.
- Tool contracts and system prompts strongly influence retrieval decisions.
- Middleware is useful for cross-cutting behavior such as logging, limits, retries, and guardrails.
- Explicit graphs are useful when the application needs custom state or control flow.
- Inspecting intermediate events is essential because a plausible final answer can hide a poor agent path.

### Further Reading

- [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain Middleware](https://docs.langchain.com/oss/python/langchain/middleware)
- [LangGraph Overview](https://docs.langchain.com/oss/python/langgraph/overview)
- [LangSmith Observability](https://docs.langchain.com/langsmith/observability)

### Notebook Output Guidance

Keep useful outputs when you submit, especially graph diagrams and representative streamed runs that support your observations. Remove secrets, failed experiments that no longer matter, and excessively noisy output.